# 🚲 Post-Deploy Setup — Ontology, Graph Model, Data Agents & Activators

This notebook deploys 7 items that `fabric-cicd` can't handle directly:

1. **Bicycle_Ontology_Model_New** (Ontology) — 12 entity types, 23 relationships
2. **Bicycle_Ontology_Model_New_graph** (GraphModel) — connected to bicycles_gold lakehouse
3. **Cycling-Campaign-Agent** (OperationsAgent) — campaign automation agent
4. **Bicycle Fleet Intelligence Agent** (DataAgent) — NL→SQL across lakehouse + SM + graph
5. **ontology data agent** (DataAgent) — graph-backed ontology reasoning (depends on ontology from step 1)
6. **BicycleFleet_Activator** (Reflex) — 4 alert rules (bike availability monitoring)
7. **Cycling Campaign Activator** (Reflex) — 3 alert rules (depends on ontology from step 1)

**Prerequisites**: Run `Deploy_Bicycle_RTI.ipynb` first (all 5 stages must succeed).

In [ ]:
# =============================================================
# CELL 1 — Setup & Discovery
# =============================================================
import json, requests, time, base64, os
import sempy.fabric as fabric
from notebookutils import mssparkutils

# Get workspace info
ws_id = fabric.get_workspace_id()
print(f"Workspace: {ws_id}")

# Get auth token
token = mssparkutils.credentials.getToken("https://analysis.windows.net/powerbi/api")
headers = {"Authorization": f"Bearer {token}", "Content-Type": "application/json"}

# Discover bicycles_gold lakehouse ID
items = fabric.list_items()
gold_lh = items[items['Display Name'] == 'bicycles_gold']
if gold_lh.empty:
    raise RuntimeError("bicycles_gold lakehouse not found! Run Deploy_Bicycle_RTI.ipynb first.")
gold_lh_id = gold_lh.iloc[0]['Id']
print(f"bicycles_gold lakehouse: {gold_lh_id}")

# Discover Bicycle Ontology Model SM ID
ont_sm = items[items['Display Name'] == 'Bicycle Ontology Model']
ont_sm_id = ont_sm.iloc[0]['Id'] if not ont_sm.empty else None
print(f"Bicycle Ontology Model SM: {ont_sm_id}")

print("\n✅ Discovery complete")

In [ ]:
# =============================================================
# CELL 2 — Deploy Ontology (Bicycle_Ontology_Model_New)
# =============================================================
# Uses the Fabric Ontology REST API to:
#  1. Create the ontology item
#  2. Load the full definition (12 entity types, 23 relationships,
#     data bindings, contextualizations) from disk
#  3. Patch workspace/lakehouse IDs for the target environment
#  4. Push the definition via updateDefinition API
#
# API: POST /v1/workspaces/{id}/ontologies
# API: POST /v1/workspaces/{id}/ontologies/{id}/updateDefinition
# Ref: https://learn.microsoft.com/en-us/rest/api/fabric/ontology/items

import re

# --- Smart path resolver: find post_deploy/definitions/ regardless of upload location ---
_candidates = [
    "/lakehouse/default/Files/post_deploy/definitions",
    "/lakehouse/default/Files/RTI-Hackathon-Demo/post_deploy/definitions",
]
DEFINITIONS_BASE = None
for _c in _candidates:
    if os.path.exists(_c):
        DEFINITIONS_BASE = _c
        break
if DEFINITIONS_BASE:
    print(f"  Definitions folder: {DEFINITIONS_BASE}")
else:
    print("  [ERROR] Cannot find post_deploy/definitions/ on lakehouse!")
    print("          Upload the post_deploy folder to the attached lakehouse under Files/")

ONTOLOGY_NAME = "Bicycle_Ontology_Model_New"
ONT_API = f"https://api.fabric.microsoft.com/v1/workspaces/{ws_id}/ontologies"

def load_ontology_parts(base_path):
    """Walk the ontology directory and build base64-encoded parts."""
    parts = []
    for root, _dirs, files in sorted(os.walk(base_path)):
        for fname in sorted(files):
            if fname == ".platform":
                continue
            filepath = os.path.join(root, fname)
            rel_path = os.path.relpath(filepath, base_path).replace("\\", "/")
            with open(filepath, 'rb') as f:
                raw = f.read()
            if raw.startswith(b'\xef\xbb\xbf'):
                raw = raw[3:]
            payload = base64.b64encode(raw).decode("utf-8")
            parts.append({"path": rel_path, "payload": payload, "payloadType": "InlineBase64"})
    return parts

def patch_bindings(parts, target_ws_id, target_lh_id):
    """Patch workspace/lakehouse IDs in DataBinding and Contextualization parts."""
    patched = []
    for part in parts:
        path = part["path"]
        if "DataBindings/" in path or "Contextualizations/" in path:
            try:
                content = base64.b64decode(part["payload"]).decode("utf-8")
                obj = json.loads(content)
                _patch_ids(obj, target_ws_id, target_lh_id)
                content = json.dumps(obj, indent=2, ensure_ascii=False)
                part = {**part, "payload": base64.b64encode(content.encode("utf-8")).decode("utf-8")}
            except Exception as e:
                print(f"    [WARN] Could not patch {path}: {e}")
        patched.append(part)
    return patched

def _patch_ids(obj, ws_id, lh_id):
    if isinstance(obj, dict):
        for key in list(obj.keys()):
            val = obj[key]
            lk = key.lower()
            if lk in ("workspaceid", "workspaceguid", "workspace_id"):
                obj[key] = ws_id
            elif lk in ("itemid", "lakehouseid", "artifactid", "item_id"):
                obj[key] = lh_id
            elif isinstance(val, str) and "onelake" in val.lower():
                m = re.match(r'(abfss://)([0-9a-f-]+)(@onelake[^/]*/)([0-9a-f-]+)(.*)', val, re.I)
                if m:
                    obj[key] = f"{m.group(1)}{ws_id}{m.group(3)}{lh_id}{m.group(5)}"
            elif isinstance(val, (dict, list)):
                _patch_ids(val, ws_id, lh_id)
    elif isinstance(obj, list):
        for item in obj:
            if isinstance(item, (dict, list)):
                _patch_ids(item, ws_id, lh_id)

def wait_lro(response, label, timeout=180):
    loc = response.headers.get("Location")
    if not loc:
        time.sleep(10)
        return True
    start = time.time()
    retry = int(response.headers.get("Retry-After", 5))
    while time.time() - start < timeout:
        time.sleep(retry)
        r = requests.get(loc, headers=headers)
        if r.status_code == 200:
            status = r.json().get("status", "")
            if status == "Succeeded":
                return True
            if status in ("Failed", "Cancelled"):
                print(f"    [FAIL] {label}: {status}")
                return False
    return False

print(f"Deploying Ontology: {ONTOLOGY_NAME}")
print(f"  Workspace: {ws_id}")
print(f"  Lakehouse: {gold_lh_id}")

# Check if already exists (try items DF first, then API)
existing_ont = items[items['Display Name'] == ONTOLOGY_NAME]
if not existing_ont.empty:
    ont_id = existing_ont.iloc[0]['Id']
    print(f"  Already exists (ID: {ont_id}) — will update definition")
else:
    # Create empty ontology via dedicated API
    create_body = {
        "displayName": ONTOLOGY_NAME,
        "description": "Bicycle Fleet Ontology — 12 entity types, 23 relationships, bound to bicycles_gold",
    }
    r = requests.post(ONT_API, headers=headers, json=create_body)
    if r.status_code in (200, 201):
        ont_id = r.json().get("id", "")
        print(f"  Created: {ont_id}")
    elif r.status_code == 202:
        wait_lro(r, "create ontology")
        time.sleep(3)
        r2 = requests.get(ONT_API, headers=headers)
        ont_id = None
        for o in r2.json().get("value", []):
            if o["displayName"] == ONTOLOGY_NAME:
                ont_id = o["id"]
                break
        print(f"  Created (async): {ont_id}")
    elif r.status_code == 400 and "AlreadyInUse" in r.text:
        # Item was created in a previous run but items DF is stale — look it up
        print(f"  Already exists (stale cache) — looking up ID via API...")
        r2 = requests.get(ONT_API, headers=headers)
        ont_id = None
        for o in r2.json().get("value", []):
            if o["displayName"] == ONTOLOGY_NAME:
                ont_id = o["id"]
                break
        if ont_id:
            print(f"  Found existing: {ont_id} — will update definition")
        else:
            print(f"  [FAIL] Item exists but could not find via API")
    else:
        print(f"  [FAIL] Create: {r.status_code} {r.text[:300]}")
        ont_id = None

if ont_id:
    # Load definition from disk
    ont_dir = f"{DEFINITIONS_BASE}/Bicycle_Ontology_Model_New.Ontology" if DEFINITIONS_BASE else ""
    if not ont_dir or not os.path.exists(ont_dir):
        print(f"  [WARN] Ontology definition dir not found")
        print(f"         Upload post_deploy/ folder to lakehouse Files/ first")
    else:
        parts = load_ontology_parts(ont_dir)
        et_count = sum(1 for p in parts if p["path"].startswith("EntityTypes/") and p["path"].endswith("/definition.json"))
        rt_count = sum(1 for p in parts if p["path"].startswith("RelationshipTypes/") and p["path"].endswith("/definition.json"))
        db_count = sum(1 for p in parts if "DataBindings/" in p["path"])
        ctx_count = sum(1 for p in parts if "Contextualizations/" in p["path"])
        print(f"  Loaded {len(parts)} parts: {et_count} entities, {rt_count} relationships, {db_count} bindings, {ctx_count} contextualizations")

        # Patch IDs for target environment
        parts = patch_bindings(parts, ws_id, gold_lh_id)
        print(f"  Patched data bindings for target environment")

        # Push full definition
        update_url = f"{ONT_API}/{ont_id}/updateDefinition"
        body = {"definition": {"parts": parts}}
        r = requests.post(update_url, headers=headers, json=body)
        if r.status_code in (200, 201):
            print(f"  ✅ Definition pushed successfully")
        elif r.status_code == 202:
            ok = wait_lro(r, "updateDefinition")
            print(f"  ✅ Definition pushed successfully" if ok else "  [FAIL] updateDefinition failed")
        else:
            print(f"  [FAIL] updateDefinition: {r.status_code} {r.text[:300]}")

        # Verify
        verify_url = f"{ONT_API}/{ont_id}/getDefinition"
        vr = requests.post(verify_url, headers=headers)
        if vr.status_code == 200:
            vparts = vr.json().get("definition", {}).get("parts", [])
            print(f"  Verified: {len(vparts)} parts in ontology definition")
        elif vr.status_code == 202:
            print(f"  Verification pending (LRO) — check in Fabric UI")

print("\n✅ Ontology step complete")


In [ ]:
# =============================================================
# CELL 3 — Deploy Graph Model
# =============================================================
# The graph model is auto-provisioned when you open the ontology in
# the Fabric UI, but we can also deploy a standalone graph model via
# the GraphModel REST API for full programmatic control.
#
# API: POST /v1/workspaces/{id}/graphModels
# API: POST /v1/workspaces/{id}/graphModels/{id}/updateDefinition
# API: POST /v1/workspaces/{id}/graphModels/{id}/jobs/refreshGraph/instances
# Ref: https://learn.microsoft.com/en-us/rest/api/fabric/graphmodel/items
#
# NOTE: Graph refresh for ontology-managed graphs returns InvalidJobType.
#       Only the Fabric UI 'Refresh now' button works for auto-provisioned
#       graphs. Standalone graphs CAN be refreshed via API.

GRAPH_MODEL_NAME = "Bicycle_Ontology_Model_New_graph"
GM_API = f"https://api.fabric.microsoft.com/v1/workspaces/{ws_id}/graphModels"

print(f"Deploying GraphModel: {GRAPH_MODEL_NAME}")

# Check if already exists
existing_gm = items[items['Display Name'].str.contains(GRAPH_MODEL_NAME, na=False)]
if not existing_gm.empty:
    gm_id = existing_gm.iloc[0]['Id']
    print(f"  Already exists (ID: {gm_id}) — will update definition")
else:
    gm_id = None

# Load definition from disk
gm_dir = f"{DEFINITIONS_BASE}/Bicycle_Ontology_Model_New_graph.GraphModel" if DEFINITIONS_BASE else ""
gm_files = ['graphType.json', 'dataSources.json', 'graphDefinition.json', 'stylingConfiguration.json']

gm_parts = []
if os.path.exists(gm_dir):
    for gm_file in gm_files:
        filepath = os.path.join(gm_dir, gm_file)
        if os.path.exists(filepath):
            with open(filepath, 'r') as f:
                content = f.read()
            # Patch workspace/lakehouse placeholders
            content = content.replace("__WORKSPACE_ID__", ws_id)
            content = content.replace("__LH_BICYCLES_GOLD__", gold_lh_id)
            encoded = base64.b64encode(content.encode('utf-8')).decode('utf-8')
            gm_parts.append({"path": gm_file, "payload": encoded, "payloadType": "InlineBase64"})
            print(f"  Loaded: {gm_file}")
        else:
            print(f"  [WARN] Missing: {gm_file}")
else:
    print(f"  [WARN] Graph model dir not found at {gm_dir}")
    print(f"         Upload the repo to lakehouse Files first")

if gm_parts:
    if gm_id:
        # Update existing definition
        update_url = f"{GM_API}/{gm_id}/updateDefinition"
        body = {"definition": {"parts": gm_parts}}
        r = requests.post(update_url, headers=headers, json=body)
        if r.status_code in (200, 201):
            print(f"  ✅ Definition updated")
        elif r.status_code == 202:
            ok = wait_lro(r, "updateDefinition")
            print(f"  ✅ Definition updated" if ok else "  [FAIL] Update failed")
        else:
            print(f"  [FAIL] updateDefinition: {r.status_code} {r.text[:300]}")
    else:
        # Create new graph model with definition
        create_body = {
            "displayName": GRAPH_MODEL_NAME,
            "description": "Graph model for Bicycle Fleet Ontology — 12 node types, 23 edge types",
            "definition": {"parts": gm_parts},
        }
        r = requests.post(GM_API, headers=headers, json=create_body)
        if r.status_code in (200, 201):
            gm_id = r.json().get('id', '')
            print(f"  ✅ Created: {gm_id}")
        elif r.status_code == 202:
            ok = wait_lro(r, "create graph model")
            if ok:
                time.sleep(3)
                r2 = requests.get(GM_API, headers=headers)
                for g in r2.json().get("value", []):
                    if GRAPH_MODEL_NAME in g.get("displayName", ""):
                        gm_id = g["id"]
                        break
                print(f"  ✅ Created (async): {gm_id}")
        else:
            print(f"  [FAIL] Create: {r.status_code} {r.text[:300]}")

    # Try refresh (will fail for ontology-managed graphs)
    if gm_id:
        print(f"\n  Attempting graph refresh...")
        refresh_url = f"{GM_API}/{gm_id}/jobs/refreshGraph/instances"
        r = requests.post(refresh_url, headers=headers)
        if r.status_code in (200, 202):
            print(f"  Refresh triggered — check Fabric UI for status")
        else:
            error = r.json().get("error", {}) if r.text else {}
            code = error.get("errorCode", "")
            if "InvalidJobType" in code or "InvalidJobType" in r.text:
                print(f"  [INFO] InvalidJobType — this is an ontology-managed graph")
                print(f"         Use Fabric UI > Graph Model > 'Refresh now' button")
            else:
                print(f"  [WARN] Refresh: {r.status_code} — may need manual refresh")
else:
    print("  No definition parts loaded — skipping graph creation")

print("\n✅ Graph model step complete")

In [ ]:
# =============================================================
# CELL 4 — Deploy Operations Agent (Cycling-Campaign-Agent)
# =============================================================
# Uses the generic items API since OperationsAgent doesn't have
# a dedicated REST API endpoint yet.
# Patches __KQL_DB__ and __WORKSPACE_ID__ placeholders in
# Configurations.json before sending.

print("Creating OperationsAgent: Cycling-Campaign-Agent ...")

existing_oa = items[items['Display Name'] == 'Cycling-Campaign-Agent']
if not existing_oa.empty:
    print(f"  Already exists (ID: {existing_oa.iloc[0]['Id']}) — skipping")
else:
    # Discover KQL Database ID for the Operations Agent datasource
    kql_db_row = items[items['Display Name'] == 'bikerentaldb']
    kql_db_id = kql_db_row.iloc[0]['Id'] if not kql_db_row.empty else None
    print(f"  KQL Database (bikerentaldb): {kql_db_id or 'NOT FOUND'}")

    oa_dir = f"{DEFINITIONS_BASE}/Cycling-Campaign-Agent.OperationsAgent" if DEFINITIONS_BASE else ""
    oa_files = ['Configurations.json']

    parts = []
    if oa_dir and os.path.exists(oa_dir):
        for oa_file in oa_files:
            filepath = os.path.join(oa_dir, oa_file)
            if os.path.exists(filepath):
                with open(filepath, 'r') as f:
                    content = f.read()
                # Patch placeholders
                content = content.replace("__WORKSPACE_ID__", ws_id)
                if kql_db_id:
                    content = content.replace("__KQL_DB__", kql_db_id)
                else:
                    print("  [WARN] KQL DB not found — placeholder not patched")
                encoded = base64.b64encode(content.encode('utf-8')).decode('utf-8')
                parts.append({"path": oa_file, "payload": encoded, "payloadType": "InlineBase64"})
                print(f"  Loaded & patched: {oa_file}")
    else:
        print(f"  [WARN] Agent dir not found at {oa_dir}")

    if parts:
        # Try create with definition first
        create_body = {
            "displayName": "Cycling-Campaign-Agent",
            "type": "OperationsAgent",
            "definition": {"parts": parts},
        }
        r = requests.post(
            f"https://api.fabric.microsoft.com/v1/workspaces/{ws_id}/items",
            headers=headers,
            json=create_body
        )
        if r.status_code in (200, 201):
            oa_id = r.json().get('id', '')
            print(f"  ✅ Created: {oa_id}")
        elif r.status_code == 202:
            ok = wait_lro(r, "create operations agent")
            print(f"  ✅ Created (async)" if ok else "  [FAIL] Create failed")
        elif r.status_code == 500:
            # Fabric sometimes can't handle definition in create — try empty shell
            print(f"  [INFO] Create-with-definition returned 500 — trying empty shell...")
            create_body2 = {
                "displayName": "Cycling-Campaign-Agent",
                "type": "OperationsAgent",
            }
            r2 = requests.post(
                f"https://api.fabric.microsoft.com/v1/workspaces/{ws_id}/items",
                headers=headers,
                json=create_body2
            )
            if r2.status_code in (200, 201):
                oa_id = r2.json().get('id', '')
                print(f"  Created empty shell: {oa_id}")
                # Now push definition
                update_url = f"https://api.fabric.microsoft.com/v1/workspaces/{ws_id}/items/{oa_id}/updateDefinition"
                r3 = requests.post(update_url, headers=headers, json={"definition": {"parts": parts}})
                if r3.status_code in (200, 201):
                    print(f"  ✅ Definition pushed")
                elif r3.status_code == 202:
                    ok = wait_lro(r3, "push OA definition")
                    print(f"  ✅ Definition pushed" if ok else "  [WARN] Definition push may have failed — configure in Fabric UI")
                else:
                    print(f"  [WARN] Definition push: {r3.status_code} — configure goals/instructions in Fabric UI")
                    print(f"         The agent shell exists, just needs manual configuration")
            elif r2.status_code == 202:
                ok = wait_lro(r2, "create empty OA")
                print(f"  Created empty shell (async) — configure in Fabric UI" if ok else "  [FAIL] Empty shell create also failed")
            else:
                print(f"  [FAIL] Empty shell create: {r2.status_code} {r2.text[:300]}")
        else:
            print(f"  [FAIL] Create: {r.status_code} {r.text[:300]}")
    else:
        print("  No definition files — skipping")

print("\n✅ Operations agent step complete")


In [ ]:
# =============================================================
# CELL 5 — Deploy Bicycle Fleet Intelligence Agent (DataAgent)
# =============================================================
# Reads definition from post_deploy/definitions/ on the lakehouse,
# patches workspace, lakehouse, and SM artifact IDs, then POSTs
# via the generic items API.  Excludes graph datasource (matches
# the original manifest.json which omitted it).
# =============================================================

AGENT_NAME = "Bicycle Fleet Intelligence Agent"
AGENT_DIR  = f"{DEFINITIONS_BASE}/Bicycle Fleet Intelligence Agent.DataAgent" if DEFINITIONS_BASE else ""

# Source IDs that need patching
SRC_WORKSPACE  = "573cc7c7-a45a-4fd9-886e-9db4e9798db4"
SRC_LAKEHOUSE  = "edb7391d-0c43-56ff-981f-fd19c6a770f7"
SRC_SM         = "340a85b8-9423-55e5-afd2-896cbe7ad542"

# Discover Bicycle RTI Analytics SM ID
sm_row = items[items['Display Name'] == 'Bicycle RTI Analytics']
sm_id  = sm_row.iloc[0]['Id'] if not sm_row.empty else None
print(f"Bicycle RTI Analytics SM: {sm_id}")

def load_agent_parts(base_path, skip_graph=True):
    """Walk the agent directory and build base64-encoded parts.
    Skips .platform and optionally graph datasource (not in manifest)."""
    parts = []
    for root, _dirs, files in sorted(os.walk(base_path)):
        for fname in sorted(files):
            if fname == ".platform" or fname == "manifest.json":
                continue
            filepath = os.path.join(root, fname)
            rel = os.path.relpath(filepath, base_path).replace("\\", "/")
            # Skip graph datasource — not in original manifest.json
            if skip_graph and "graph-" in rel:
                continue
            with open(filepath, 'rb') as f:
                raw = f.read()
            if raw.startswith(b'\xef\xbb\xbf'):
                raw = raw[3:]  # strip BOM
            parts.append({"path": rel, "raw": raw, "rel": rel})
    return parts

def patch_agent_content(raw_bytes, tgt_ws, tgt_lh, tgt_sm, src_ws, src_lh, src_sm):
    """Replace hardcoded workspace / lakehouse / SM GUIDs in JSON content."""
    try:
        text = raw_bytes.decode('utf-8')
    except UnicodeDecodeError:
        return raw_bytes  # binary file — skip
    text = text.replace(src_ws, tgt_ws)
    if tgt_lh and src_lh:
        text = text.replace(src_lh, tgt_lh)
    if tgt_sm and src_sm:
        text = text.replace(src_sm, tgt_sm)
    return text.encode('utf-8')

print(f"\nDeploying DataAgent: {AGENT_NAME}")
print(f"  Workspace : {ws_id}")
print(f"  Lakehouse : {gold_lh_id}")
print(f"  SM        : {sm_id}")

existing_da = items[items['Display Name'] == AGENT_NAME]
if not existing_da.empty:
    da_id = existing_da.iloc[0]['Id']
    print(f"  Already exists (ID: {da_id}) — skipping create")
else:
    if not os.path.exists(AGENT_DIR):
        print(f"  [WARN] Agent dir not found at {AGENT_DIR}")
        da_id = None
    else:
        raw_parts = load_agent_parts(AGENT_DIR, skip_graph=True)
        encoded_parts = []
        for p in raw_parts:
            patched = patch_agent_content(
                p["raw"], ws_id, gold_lh_id, sm_id,
                SRC_WORKSPACE, SRC_LAKEHOUSE, SRC_SM
            )
            encoded_parts.append({
                "path": p["rel"],
                "payload": base64.b64encode(patched).decode('utf-8'),
                "payloadType": "InlineBase64"
            })
        print(f"  Sending {len(encoded_parts)} definition parts ...")
        for ep in encoded_parts:
            print(f"    • {ep['path']}")

        create_body = {
            "displayName": AGENT_NAME,
            "type": "DataAgent",
            "definition": {"parts": encoded_parts},
        }
        r = requests.post(
            f"https://api.fabric.microsoft.com/v1/workspaces/{ws_id}/items",
            headers=headers,
            json=create_body
        )
        if r.status_code in (200, 201):
            da_id = r.json().get('id', '')
            print(f"  ✅ Created: {da_id}")
        elif r.status_code == 202:
            ok = wait_lro(r, "create Bicycle Fleet Intelligence Agent")
            print(f"  ✅ Created (async)" if ok else "  [FAIL] Create failed")
        else:
            print(f"  [FAIL] Create: {r.status_code} {r.text[:500]}")

print("\n✅ Bicycle Fleet Intelligence Agent step complete")

In [ ]:
# =============================================================
# CELL 6 — Deploy ontology data agent (DataAgent)
# =============================================================
# Minimal agent — 3 definition files, type "ontology" datasource.
# Depends on Ontology deployed in Cell 2, and uses ont_id set there.
# Patches workspace ID + __ONTOLOGY_FLEET__ placeholder.
# =============================================================

ONT_AGENT_NAME = "ontology data agent"
ONT_AGENT_DIR  = f"{DEFINITIONS_BASE}/ontology data agent.DataAgent" if DEFINITIONS_BASE else ""

# Resolve the ontology artifact ID — use the one created in Cell 2 (Ontology step)
if 'ont_id' not in dir() or not ont_id:
    # Try to discover from workspace items
    ont_row = items[items['Display Name'] == 'Bicycle_Ontology_Model_New']
    ont_id = ont_row.iloc[0]['Id'] if not ont_row.empty else None
    print(f"Discovered Ontology ID: {ont_id}")
else:
    print(f"Using Ontology ID from Cell 2: {ont_id}")

def patch_ontology_agent_content(raw_bytes, tgt_ws, ontology_id, src_ws):
    """Replace workspace ID and __ONTOLOGY_FLEET__ placeholder."""
    try:
        text = raw_bytes.decode('utf-8')
    except UnicodeDecodeError:
        return raw_bytes
    text = text.replace(src_ws, tgt_ws)
    if ontology_id:
        text = text.replace("__ONTOLOGY_FLEET__", ontology_id)
    return text.encode('utf-8')

print(f"\nDeploying DataAgent: {ONT_AGENT_NAME}")
print(f"  Workspace  : {ws_id}")
print(f"  Ontology ID: {ont_id}")

existing_oa2 = items[items['Display Name'] == ONT_AGENT_NAME]
if not existing_oa2.empty:
    oa2_id = existing_oa2.iloc[0]['Id']
    print(f"  Already exists (ID: {oa2_id}) — skipping create")
else:
    if not ont_id:
        print("  [WARN] Ontology not deployed yet — run Cell 2 (Ontology) first!")
        oa2_id = None
    elif not os.path.exists(ONT_AGENT_DIR):
        print(f"  [WARN] Agent dir not found at {ONT_AGENT_DIR}")
        oa2_id = None
    else:
        raw_parts = load_agent_parts(ONT_AGENT_DIR, skip_graph=False)
        encoded_parts = []
        for p in raw_parts:
            patched = patch_ontology_agent_content(
                p["raw"], ws_id, ont_id, SRC_WORKSPACE
            )
            encoded_parts.append({
                "path": p["rel"],
                "payload": base64.b64encode(patched).decode('utf-8'),
                "payloadType": "InlineBase64"
            })
        print(f"  Sending {len(encoded_parts)} definition parts ...")
        for ep in encoded_parts:
            print(f"    • {ep['path']}")

        create_body = {
            "displayName": ONT_AGENT_NAME,
            "type": "DataAgent",
            "definition": {"parts": encoded_parts},
        }
        r = requests.post(
            f"https://api.fabric.microsoft.com/v1/workspaces/{ws_id}/items",
            headers=headers,
            json=create_body
        )
        if r.status_code in (200, 201):
            oa2_id = r.json().get('id', '')
            print(f"  ✅ Created: {oa2_id}")
        elif r.status_code == 202:
            ok = wait_lro(r, "create ontology data agent")
            print(f"  ✅ Created (async)" if ok else "  [FAIL] Create failed")
        else:
            print(f"  [FAIL] Create: {r.status_code} {r.text[:500]}")

print("\n✅ ontology data agent step complete")

In [ ]:
# =============================================================
# CELL 7 — Deploy BicycleFleet Activator (Reflex)
# =============================================================
# The BicycleFleet_Activator monitors bike availability in real-time.
# fabric-cicd doesn't support the Reflex item type, so we deploy via
# the Items REST API instead.
#
# 4 Rules:
#   - Empty Station (Teams) — triggers when No_Bikes == 0
#   - Full Station (Teams) — triggers when No_Empty_Docks == 0
#   - Low Availability (Email) — triggers when No_Bikes < 3
#   - High Demand (Teams) — triggers when No_Empty_Docks < 3
#
# __ALERT_RECIPIENT_EMAIL__ is replaced with the deploying user's email.
# Source workspace GUID is replaced with the target workspace.

import base64

BF_ACTIVATOR_NAME = "BicycleFleet_Activator"
SOURCE_WORKSPACE_ID = "573cc7c7-a45a-4fd9-886e-9db4e9798db4"

# Auto-detect user email from token
def _get_email():
    try:
        tok = mssparkutils.credentials.getToken("https://api.fabric.microsoft.com")
        payload = tok.split(".")[1]
        payload += "=" * (-len(payload) % 4)
        claims = json.loads(base64.b64decode(payload))
        return claims.get("upn") or claims.get("unique_name") or claims.get("preferred_username", "")
    except Exception:
        return ""

bf_email = _get_email()
print(f"📧 Alert email: {bf_email or '(not detected — placeholder will remain)'}")

# Check if already exists
existing_bf = items[items['Display Name'] == BF_ACTIVATOR_NAME] if 'Display Name' in items.columns else pd.DataFrame()
if not existing_bf.empty:
    print(f"ℹ️  '{BF_ACTIVATOR_NAME}' already exists — skipping")
else:
    # Find the Reflex definition in lakehouse
    bf_dir = None
    for candidate in [
        f"/lakehouse/default/Files/workspace/{BF_ACTIVATOR_NAME}.Reflex",
        f"/lakehouse/default/Files/RTI-Hackathon-Demo/workspace/{BF_ACTIVATOR_NAME}.Reflex",
        f"/lakehouse/default/Files/RTI-Hackathon-Demo-main/workspace/{BF_ACTIVATOR_NAME}.Reflex",
    ]:
        if os.path.isdir(candidate):
            bf_dir = candidate
            break

    if not bf_dir:
        print(f"⚠️ Could not find '{BF_ACTIVATOR_NAME}.Reflex' folder in lakehouse")
        print("   Expected in: /lakehouse/default/Files/workspace/")
    else:
        entities_path = os.path.join(bf_dir, "ReflexEntities.json")
        if os.path.exists(entities_path):
            with open(entities_path, "r", encoding="utf-8") as f:
                bf_content = f.read()

            # Patch placeholders
            patches = 0
            if ws_id != SOURCE_WORKSPACE_ID and SOURCE_WORKSPACE_ID in bf_content:
                bf_content = bf_content.replace(SOURCE_WORKSPACE_ID, ws_id)
                patches += 1
                print(f"   🔄 Replaced source workspace GUID → {ws_id}")

            if bf_email and "__ALERT_RECIPIENT_EMAIL__" in bf_content:
                bf_content = bf_content.replace("__ALERT_RECIPIENT_EMAIL__", bf_email)
                patches += 1
                print(f"   📧 Replaced __ALERT_RECIPIENT_EMAIL__ → {bf_email}")

            print(f"   Applied {patches} patch(es)")

            # Encode and create via REST API
            bf_b64 = base64.b64encode(bf_content.encode("utf-8")).decode("utf-8")
            create_body = {
                "displayName": BF_ACTIVATOR_NAME,
                "type": "Reflex",
                "definition": {
                    "parts": [
                        {
                            "path": "ReflexEntities.json",
                            "payload": bf_b64,
                            "payloadType": "InlineBase64"
                        }
                    ]
                }
            }

            r = requests.post(
                f"https://api.fabric.microsoft.com/v1/workspaces/{ws_id}/items",
                headers=headers,
                json=create_body
            )

            if r.status_code in (200, 201):
                bf_id = r.json().get("id", "")
                print(f"✅ Created '{BF_ACTIVATOR_NAME}': {bf_id}")
            elif r.status_code == 202:
                ok = wait_lro(r, f"create {BF_ACTIVATOR_NAME}")
                print(f"✅ Created '{BF_ACTIVATOR_NAME}' (async)" if ok else f"❌ Failed to create '{BF_ACTIVATOR_NAME}'")
            else:
                print(f"❌ Failed to create '{BF_ACTIVATOR_NAME}': HTTP {r.status_code}")
                print(f"   {r.text[:500]}")
        else:
            print(f"⚠️ ReflexEntities.json not found in {bf_dir}")

print("\n✅ BicycleFleet Activator step complete")

In [ ]:
# =============================================================
# CELL 8 — Deploy Cycling Campaign Activator (Reflex)
# =============================================================
# The Cycling Campaign Activator depends on:
#   1. The Ontology (created in Cell 2) — __ONTOLOGY_NEW__ placeholder
#   2. Power Automate flow (per-user) — kept as-is, user connects manually
#   3. User's email — __ALERT_RECIPIENT_EMAIL__ placeholder
#
# This cell:
#   a. Reads the ReflexEntities.json from the lakehouse
#   b. Replaces __ONTOLOGY_NEW__ with the actual ontology item ID
#   c. Replaces source workspace GUID with target
#   d. Replaces __ALERT_RECIPIENT_EMAIL__ with deploying user's email
#   e. Creates the Activator via the Items REST API
#
# NOTE: The Power Automate flow action will need manual connection
# after deployment. See README for Power Automate Developer Plan setup.

import base64

ACTIVATOR_NAME = "Cycling Campaign Activator"
SOURCE_WORKSPACE_ID = "573cc7c7-a45a-4fd9-886e-9db4e9798db4"

# Auto-detect user email from token
def _get_alert_email():
    try:
        tok = notebookutils.credentials.getToken("https://api.fabric.microsoft.com")
        payload = tok.split(".")[1]
        payload += "=" * (-len(payload) % 4)
        claims = json.loads(base64.b64decode(payload))
        return claims.get("upn") or claims.get("unique_name") or claims.get("preferred_username", "")
    except Exception:
        return ""

alert_email = _get_alert_email()
print(f"📧 Alert email: {alert_email or '(not detected — placeholder will remain)'}")

# Find the deployed ontology ID (created in Cell 2)
ontology_id = None
resp = requests.get(f"https://api.fabric.microsoft.com/v1/workspaces/{ws_id}/items",
                    headers=headers)
if resp.status_code == 200:
    for item in resp.json().get("value", []):
        if item.get("displayName") == "Bicycle_Ontology_Model_New" and item.get("type") == "Ontology":
            ontology_id = item["id"]
            break

if ontology_id:
    print(f"✅ Found Ontology: {ontology_id}")
else:
    print("⚠️ Ontology 'Bicycle_Ontology_Model_New' not found — run Cell 2 first")
    print("   __ONTOLOGY_NEW__ placeholder will remain (Activator won't connect to ontology)")

# Check if already exists
existing = items[items['Display Name'] == ACTIVATOR_NAME] if 'Display Name' in items.columns else pd.DataFrame()
if not existing.empty:
    print(f"ℹ️  '{ACTIVATOR_NAME}' already exists — skipping")
else:
    # Read the Reflex definition from lakehouse
    reflex_dir = None
    for candidate in [
        f"/lakehouse/default/Files/workspace/{ACTIVATOR_NAME}.Reflex",
        f"/lakehouse/default/Files/RTI-Hackathon-Demo/workspace/{ACTIVATOR_NAME}.Reflex",
        f"/lakehouse/default/Files/RTI-Hackathon-Demo-main/workspace/{ACTIVATOR_NAME}.Reflex",
    ]:
        if os.path.isdir(candidate):
            reflex_dir = candidate
            break

    if not reflex_dir:
        print(f"⚠️ Could not find '{ACTIVATOR_NAME}.Reflex' folder in lakehouse")
        print("   Expected in: /lakehouse/default/Files/workspace/")
    else:
        # Read ReflexEntities.json
        entities_path = os.path.join(reflex_dir, "ReflexEntities.json")
        if os.path.exists(entities_path):
            with open(entities_path, "r", encoding="utf-8") as f:
                entities_content = f.read()

            # Patch placeholders
            patches = 0
            if ontology_id and "__ONTOLOGY_NEW__" in entities_content:
                entities_content = entities_content.replace("__ONTOLOGY_NEW__", ontology_id)
                patches += 1
                print(f"   🔄 Replaced __ONTOLOGY_NEW__ → {ontology_id}")

            if ws_id != SOURCE_WORKSPACE_ID and SOURCE_WORKSPACE_ID in entities_content:
                entities_content = entities_content.replace(SOURCE_WORKSPACE_ID, ws_id)
                patches += 1
                print(f"   🔄 Replaced source workspace GUID → {ws_id}")

            if alert_email and "__ALERT_RECIPIENT_EMAIL__" in entities_content:
                entities_content = entities_content.replace("__ALERT_RECIPIENT_EMAIL__", alert_email)
                patches += 1
                print(f"   📧 Replaced __ALERT_RECIPIENT_EMAIL__ → {alert_email}")

            print(f"   Applied {patches} patch(es)")

            # Encode for the API
            entities_b64 = base64.b64encode(entities_content.encode("utf-8")).decode("utf-8")

            # Create the Reflex item
            create_body = {
                "displayName": ACTIVATOR_NAME,
                "type": "Reflex",
                "definition": {
                    "parts": [
                        {
                            "path": "ReflexEntities.json",
                            "payload": entities_b64,
                            "payloadType": "InlineBase64"
                        }
                    ]
                }
            }

            r = requests.post(
                f"https://api.fabric.microsoft.com/v1/workspaces/{ws_id}/items",
                headers=headers,
                json=create_body
            )

            if r.status_code in (200, 201):
                act_id = r.json().get("id", "")
                print(f"✅ Created '{ACTIVATOR_NAME}': {act_id}")
            elif r.status_code == 202:
                ok = wait_lro(r, f"create {ACTIVATOR_NAME}")
                print(f"✅ Created '{ACTIVATOR_NAME}' (async)" if ok else f"❌ Failed to create '{ACTIVATOR_NAME}'")
            else:
                print(f"❌ Failed to create '{ACTIVATOR_NAME}': HTTP {r.status_code}")
                print(f"   {r.text[:500]}")
        else:
            print(f"⚠️ ReflexEntities.json not found in {reflex_dir}")

print("\n✅ Cycling Campaign Activator step complete")
print("   ⚠️ Power Automate flow must be connected manually in Fabric UI")
print("   See README → 'Power Automate Setup' section")

## ✅ Post-Deploy Complete!

### What was deployed (7 items)

| Item | API | Status |
|------|-----|--------|
| **Bicycle_Ontology_Model_New** | `POST /ontologies` + `updateDefinition` | 12 entity types, 23 relationships, data bindings |
| **Bicycle_Ontology_Model_New_graph** | `POST /graphModels` + `updateDefinition` | 4-part definition (graphType, dataSources, graphDefinition, styling) |
| **Cycling-Campaign-Agent** | `POST /items` | Operations agent with campaign instructions |
| **Bicycle Fleet Intelligence Agent** | `POST /items` (DataAgent) | NL→SQL across lakehouse + SM (10 definition parts) |
| **ontology data agent** | `POST /items` (DataAgent) | Graph-backed ontology reasoning (3 definition parts) |
| **BicycleFleet_Activator** | `POST /items` (Reflex) | 4 rules: Empty Station, Full Station, Low Availability, High Demand |
| **Cycling Campaign Activator** | `POST /items` (Reflex) | 3 rules: High Demand Forecast, Station Critical, Cycling Campaign (Power Automate) |

### Remaining Manual Steps

1. **Refresh Graph Model**: Open `Bicycle_Ontology_Model_New_graph` in Fabric UI → click **Refresh now**
   - API refresh returns `InvalidJobType` for ontology-managed graphs
   - This is a known limitation — only the UI button works
2. **Connect Power Automate flow**: Open `Cycling Campaign Activator` → open the "Cycling Campaign" rule → link to your Power Automate flow
   - Requires Power Automate Developer Plan (free) — see README
3. **Connect BicycleFleet_Activator to eventstream**: Open `RTIbikeRental` eventstream → add an Activator destination → select `BicycleFleet_Activator`
4. **Start Eventstreams**: Open `RTIbikeRental` and `RTI-WeatherDemo` → verify they are streaming
5. **Run the Pipeline**: Open `PL_BicycleRTI_Medallion` → Run (first load ~15-25 min)
6. **Test Data Agents**: Open `Bicycle Fleet Intelligence Agent` → ask: *"What are the top 5 busiest stations?"* | Open `ontology data agent` → ask: *"Show me stations in Sands End"*